# Proyecto Parte 2: Clasificación de sentimiento multilingüe con Deep Learning y aumentación de datos

## Miembros del grupo 35

- Juan Diego Ramírez Garzón - 202410855
- Sebastián Vargas Otero - 202323732
- Mateo Zambrano Burbano - 202321531
- Nicolás Mario Romero Colmenares - 202321309

### Resumen del enfoque

| Etapa | Decisión |
|---|---|
| Framework | TensorFlow + Keras 3 + **KerasHub** (modelos preentrenados en Keras) |
| Modelo | **XLM-RoBERTa** (*encoder-only*, preentrenado en 100 idiomas): tamaño *base* para comparar configuraciones y tamaño ***large*** para el modelo final |
| Modelo final | **Ensamble E8**: promedio de probabilidades de dos XLM-R large entrenados con semillas distintas (validación 0.9096, Kaggle público 0.89390) |
| Líneas de comparación | TF-IDF + regresión logística (modelo clásico, como en la Parte 1) y BiLSTM entrenada desde cero (sin preentrenamiento) |
| Aumentación 1 | **Code-switching léxico**: sustitución de palabras es→en y en→es con los diccionarios bilingües MUSE (no se usa ningún modelo generativo) |
| Aumentación 2 | **Inserción de oraciones neutras de otro idioma**: se intercalan oraciones descriptivas (sin opinión) tomadas de reseñas neutrales en otro idioma |
| Restricción | Por el anuncio del 12/11/2026 solo se usan modelos *encoder-only* (XLM-R) con cabeza de clasificación, y ningún modelo generativo, ni siquiera para aumentar datos |
| Validación | Partición *hold-out* 90/10 estratificada por **idioma × clase**; la aumentación solo se aplica a la partición de entrenamiento |
| Métrica | Exactitud (*accuracy*), igual que Kaggle, global y **por idioma** |

### Cómo ejecutarlo

1. Poner los archivos `train.csv`, `eval.csv` y `sample_submission.csv` en una carpeta `Data/` junto al notebook.
   - **En Colab:** subir la carpeta `Data/` al panel de archivos (queda en `/content/Data`), o montar Drive y cambiar `DATA` y `TRABAJO` en la celda de configuración.
2. Usar un entorno con **GPU** (en Colab: *Runtime → Change runtime type → T4 GPU*) y con **Internet** (para descargar los modelos preentrenados y los diccionarios MUSE).
3. Ejecutar todo en orden (*Run all*). Las predicciones de cada experimento se guardan en `resultados/`; si la sesión se corta, al volver a ejecutar se reutilizan los experimentos ya terminados (ver `REUTILIZAR_CACHE`).

Tiempos medidos con GPU T4 de Colab: aumentación < 1 min, cada BiLSTM ≈ 1 min, cada XLM-R base ≈ 35-55 min y cada XLM-R large (1 época) ≈ 1-2 horas. Total ≈ 5 horas. Con `REUTILIZAR_CACHE = True` y las salidas guardadas en Drive, volver a ejecutar el notebook se salta los entrenamientos ya hechos.

### Contenido

0. Configuración del entorno
1. Carga y calidad de datos
2. Exploración: idiomas, clases, longitud y tipos de reseña difíciles
3. Protocolo de validación
4. Aumentación de datos (code-switching léxico e inserción de oraciones neutras)
5. Experimentos (BiLSTM y XLM-RoBERTa, con y sin aumentación)
6. Comparación con y sin aumentación
7. Análisis de errores por idioma
8. Modelo final, guardado y envío
9. Conclusiones

### **0. Configuración del entorno**

In [ ]:
import os, re, gc, json, math, time, random, shutil, warnings, subprocess, sys
from pathlib import Path
from collections import Counter

os.environ.setdefault('TF_GPU_ALLOCATOR', 'cuda_malloc_async')   # libera la memoria de la GPU entre modelos (debe ir antes de importar TensorFlow)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from scipy.stats import binomtest
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 200)

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)

CLASES = ['negativo', 'neutral', 'positivo']
C2I = {c: i for i, c in enumerate(CLASES)}

# Rutas: carpeta con los CSV y carpeta donde se guardan las salidas
EN_COLAB = 'google.colab' in sys.modules or Path('/content').exists() and Path('/content/sample_data').exists()
if EN_COLAB:
    # En Colab todo se guarda en Google Drive (carpeta proyecto_ml con la subcarpeta Data) para no perder el trabajo si se reinicia la sesión
    from google.colab import drive
    drive.mount('/content/drive')
    TRABAJO = Path('/content/drive/MyDrive/proyecto_ml')
    DATA = TRABAJO / 'Data'
    DIR_MOD = Path('/content/modelos')            # pesos pesados en el disco local de Colab: no llenan Drive
    DIR_ENTREGA = TRABAJO / 'modelos_entrega'     # copia en Drive de los pesos que se necesitan para el modelo final
else:
    DATA = Path('Data')
    TRABAJO = Path('.')
    DIR_MOD = TRABAJO / 'modelos'
    DIR_ENTREGA = DIR_MOD                         # fuera de Colab no hace falta una segunda copia

DIR_AUG = TRABAJO / 'aumentacion'
DIR_ENV = TRABAJO / 'envios'
DIR_RES = TRABAJO / 'resultados'
for d in (DIR_AUG, DIR_MOD, DIR_ENV, DIR_RES, DIR_ENTREGA):
    d.mkdir(parents=True, exist_ok=True)
assert DATA.exists(), f'No se encuentra la carpeta de datos: {DATA.resolve()}. Debe contener los CSV del proyecto.'

# Si ya existen predicciones guardadas de experimentos anteriores, se reutilizan en lugar de recalcularlas
REUTILIZAR_CACHE = True
print('Datos en:', DATA.resolve())

In [ ]:
import tensorflow as tf

GPUS = tf.config.list_physical_devices('GPU')
for g in GPUS:
    tf.config.experimental.set_memory_growth(g, True)   # TF reserva memoria de GPU a medida que la necesita

def textos_tf(textos):
    """Keras 3 no acepta arreglos NumPy de strings (dtype '<U...'), así que los textos se pasan como tensor tf.string."""
    if isinstance(textos, tf.Tensor):
        return textos
    return tf.constant([str(t) for t in textos], dtype=tf.string)

import keras
try:
    import keras_hub
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'keras-hub'], check=True)
    import keras_hub

keras.utils.set_random_seed(SEMILLA)
# Precisión mixta: casi duplica la velocidad en GPUs con tensor cores (T4) y reduce memoria
keras.mixed_precision.set_global_policy('mixed_float16' if GPUS else 'float32')

print('Python', sys.version.split()[0], '| TensorFlow', tf.__version__, '| Keras', keras.__version__, '| KerasHub', keras_hub.__version__)
print('GPUs:', GPUS)

### **1. Carga y calidad de datos**

Se repite la auditoría de la Parte 1 (nulos, duplicados, etiquetas en conflicto y fuga entre `train` y `eval`), porque el conjunto de entrenamiento creció de 12,000 a 20,466 reseñas y ahora mezcla idiomas.

In [ ]:
train_raw = pd.read_csv(DATA / 'train.csv')
eval_raw = pd.read_csv(DATA / 'eval.csv')
sample = pd.read_csv(DATA / 'sample_submission.csv')
print('train:', train_raw.shape, '| eval:', eval_raw.shape)
display(train_raw.head())

In [ ]:
def clave(s):
    return s.fillna('').str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()

train = train_raw.copy()
train['label'] = train['label'].astype(str).str.strip().str.lower()
train['clave'] = clave(train['text'])
eval_df = eval_raw.copy()
eval_df['text'] = eval_df['text'].fillna('')

grupos = train.groupby('clave')['label'].agg(['size', 'nunique'])
print('Nulos train:', train_raw.isna().sum().to_dict(), '| nulos eval:', eval_raw.isna().sum().to_dict())
print('Etiquetas fuera de las 3 clases:', sorted(set(train['label']) - set(CLASES)) or 'ninguna')
print('Duplicados (clave laxa):', int(train['clave'].duplicated().sum()))
print('Textos repetidos con etiquetas distintas:', int((grupos['nunique'] > 1).sum()))
print('Reseñas de eval que aparecen en train:', int(clave(eval_df['text']).isin(train['clave']).sum()))

# Limpieza: se quitan conflictos y duplicados para evitar fuga entre train y validación
n0 = len(train)
train = train[train['clave'] != '']
train = train[~train['clave'].isin(grupos[grupos['nunique'] > 1].index)]
train = train.drop_duplicates('clave').drop(columns='clave').reset_index(drop=True)
train['y'] = train['label'].map(C2I)
print(f'train: {n0:,} -> {len(train):,} filas')

### **2. Exploración**

#### 2.1 Detección de idioma

El idioma no viene en los datos, pero lo necesitamos para (i) estratificar la validación, (ii) decidir qué reseñas aumentar y (iii) analizar los errores por idioma, como pide el enunciado.

Se usa un detector propio basado en **palabras funcionales** (artículos, preposiciones, conectores) propias de cada idioma. Se eligió así porque no requiere descargar nada, es transparente y para cuatro idiomas cercanos funciona bien. Una reseña se marca como **`es-en` (code-switching)** cuando tiene al menos 3 palabras funcionales de español y 3 de inglés, y el idioma minoritario aporta al menos el 20 %.

*Limitación:* es una heurística. Se valida manualmente con ejemplos más abajo.

In [ ]:
PALABRAS_IDIOMA = {
    'es': 'el los las que y es por para con muy pero como más mi del ya está fue eso todo cuando lo se me una al le hasta desde porque también llegó compré',
    'en': 'the and is it to of for that was with my but not this have be so you its just very really i bought arrived works great',
    'pt': 'os em um é não do da na mas muito meu minha pra eu tem foi isso chegou comprei também uma até depois veio',
    'fr': "le les et est une pour pas du des je mais très il mon ce qui avec sur ça j'ai acheté arrivé",
}
PALABRAS_IDIOMA = {k: set(v.split()) for k, v in PALABRAS_IDIOMA.items()}
TOKEN_RE = re.compile(r"[a-záéíóúàâçêôãõüñèëîïû']+")

def detectar_idioma(texto):
    palabras = TOKEN_RE.findall(texto.lower())
    c = {k: sum(p in s for p in palabras) for k, s in PALABRAS_IDIOMA.items()}
    es, en = c['es'], c['en']
    if es >= 3 and en >= 3 and min(es, en) / (es + en) >= 0.2:
        return 'es-en'
    return max(c, key=c.get)

IDIOMAS = ['es', 'en', 'pt', 'fr', 'es-en']
train['idioma'] = train['text'].map(detectar_idioma)
eval_df['idioma'] = eval_df['text'].map(detectar_idioma)

dist = pd.concat([train['idioma'].value_counts(normalize=True).rename('train'),
                  eval_df['idioma'].value_counts(normalize=True).rename('eval')], axis=1).reindex(IDIOMAS).fillna(0)
display((dist * 100).round(1).style.format('{:.1f} %').set_caption('Distribución de idiomas'))

for idi in IDIOMAS:
    print(f'== {idi}')
    for t in train.loc[train['idioma'] == idi, 'text'].sample(min(2, (train['idioma'] == idi).sum()), random_state=1):
        print('  -', t[:160], '...')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
(dist * 100).plot.bar(ax=axes[0], rot=0, color=['#4C72B0', '#DD8452'])
axes[0].set_title('Idiomas: train vs eval (%)')

pd.crosstab(train['idioma'], train['label'], normalize='index').reindex(IDIOMAS).plot.bar(
    stacked=True, ax=axes[1], rot=0, color=['#E74C3C', '#95A5A6', '#2ECC71'])
axes[1].set_title('Clases por idioma (train)')
axes[1].legend(loc='lower right', fontsize=8)

train['n_palabras'] = train['text'].str.split().str.len()
eval_df['n_palabras'] = eval_df['text'].str.split().str.len()
sns.kdeplot(train['n_palabras'], ax=axes[2], label='train')
sns.kdeplot(eval_df['n_palabras'], ax=axes[2], label='eval')
axes[2].set_title('Longitud (palabras)')
axes[2].legend()
plt.tight_layout(); plt.show()

display(train['label'].value_counts().reindex(CLASES).to_frame('n'))

#### 2.2 Tipos de reseña difíciles

El enunciado advierte de reseñas con ironía, negaciones de largo alcance, comparaciones, veredicto al inicio o que solo describen consecuencias. Se marcan con expresiones regulares multilingües (aproximadas) para comparar su frecuencia en `train` y `eval`, y luego para medir el error en cada tipo (sección 7).

In [ ]:
PATRONES = {
    'negacion_larga': r"\b(no (diría|diria|creo que|es que|puedo decir)|tampoco|i wouldn'?t say|i can'?t say|not that it|não diria|nao diria|não é que|je ne dirais|ce n'est pas que)",
    'comparacion':    r"\b(esperaba|expected|esperava|attendais|mejor que|peor que|better than|worse than|melhor que|pior que|mieux que|moins bien que)",
    'consecuencia':   r"(devolv|reembols|returned it|refund|sent it back|send it back|devolvi|rendu|renvoy|rembours)",
    'ironia':         r"(qué maravilla|que maravilla|claro que sí|just great|oh great|yeah right|que maravilha|quelle merveille|bravo|genial, |perfecto, |great, )",
    'contraste':      r"\b(pero|aunque|sin embargo|but|however|although|porém|contudo|cependant|pourtant)\b",
}

def marcar_patrones(df):
    t = df['text'].str.lower()
    return pd.DataFrame({k: t.str.contains(p, regex=True) for k, p in PATRONES.items()})

pat_train, pat_eval = marcar_patrones(train), marcar_patrones(eval_df)
tabla = pd.DataFrame({'train %': pat_train.mean() * 100, 'eval %': pat_eval.mean() * 100}).round(1)
display(tabla)
display((pd.concat([pat_train, train['label']], axis=1).groupby('label').mean() * 100).round(1).T
        .rename_axis('% de reseñas con el patrón, por clase'))

**Conclusiones de la exploración**

1. **Datos limpios:** `train` no tiene textos duplicados ni etiquetas en conflicto, y ninguna reseña de `eval` aparece en `train`, así que no hay fuga entre los conjuntos. Se conservan las 20,466 reseñas.
2. **Clases casi balanceadas** (positivo 36.6 %, negativo 36.1 %, neutral 27.4 %): la exactitud es una métrica adecuada y no hace falta remuestrear.
3. **Cambio de distribución de idiomas:** `eval` tiene menos español (61.2 % frente a 67.1 % en `train`) y más inglés (13.7 % vs 11.5 %), portugués (12.8 % vs 10.9 %) y francés (11.9 % vs 10.2 %). Un modelo ajustado al español perdería exactitud en `eval`. Por eso usamos un modelo **multilingüe preentrenado** y reportamos también una exactitud ponderada a la distribución de idiomas de `eval` (sección 3).
4. **El code-switching es escaso en los dos conjuntos:** el detector encuentra 61 reseñas `es-en` en `train` (0.3 %) y 33 en `eval` (0.5 %). Es una cota inferior, porque el detector exige al menos 3 palabras funcionales de cada idioma y no marca mezclas cortas. La falta de ejemplos es lo que motiva la aumentación A, pero también indica que este grupo pesa poco en la exactitud final.
5. **`eval` es más difícil que `train`:** las reseñas son algo más largas (61.3 vs 57.5 palabras en promedio) y los patrones difíciles son más frecuentes: negaciones de largo alcance 6.3 % vs 3.0 %, consecuencias 5.9 % vs 3.0 %, comparaciones 8.9 % vs 6.7 % y contrastes 19.0 % vs 13.4 %. Esto favorece a un modelo que lee la secuencia completa con contexto (*transformer*) frente a una bolsa de palabras, y anticipa que el score de Kaggle quedará algo por debajo de la exactitud de validación.

### **3. Protocolo de validación**

- Partición **hold-out 90 % / 10 %**, estratificada por **idioma × clase**, así la validación tiene todos los idiomas en la misma proporción que train. Se usa *hold-out* y no validación cruzada porque un *fine-tuning* de XLM-R tarda decenas de minutos. Con ~2,000 reseñas de validación el error estándar de la exactitud es de ≈ 0.6 puntos: diferencias menores a eso entre dos experimentos pueden ser ruido (por eso se usa también la prueba de McNemar).
- **La aumentación se genera solo a partir de la partición de entrenamiento.** Si una copia modificada de una reseña de validación entrara al entrenamiento, el modelo vería su contenido y la exactitud de validación quedaría inflada (fuga de información).
- Además de la exactitud global se reporta una **exactitud ponderada a la distribución de idiomas de eval**: Σ p_eval(idioma) · acc_val(idioma). Es una estimación más realista del puntaje de Kaggle, porque eval tiene más reseñas no españolas.

In [ ]:
estrato = train['idioma'] + '_' + train['label']
estrato = estrato.where(estrato.map(estrato.value_counts()) >= 2, 'raro')   # combinaciones con 1 sola reseña
tr, va = train_test_split(train, test_size=0.10, stratify=estrato, random_state=SEMILLA)
tr, va = tr.reset_index(drop=True), va.reset_index(drop=True)

P_EVAL = eval_df['idioma'].value_counts(normalize=True)
print(f'entrenamiento: {len(tr):,} | validación: {len(va):,}')
display(pd.crosstab(va['idioma'], va['label'], margins=True).reindex(IDIOMAS + ['All']))

### **4. Aumentación de datos**

#### 4.1 Qué estrategias se consideraron

El anuncio del curso del 12/11/2026 aclara que **no se permiten arquitecturas *decoder-only* ni modelos generativos de texto**. Por eso se descartó la traducción automática con modelos neuronales (MarianMT, NLLB, etc., que son generativos *encoder-decoder*) y la generación de reseñas con LLMs. Las dos estrategias elegidas **no generan texto con ningún modelo**: recombinan texto real del conjunto de entrenamiento y usan diccionarios bilingües publicados.

| Estrategia | ¿Se usa? | Razón |
|---|---|---|
| **A. Code-switching léxico con diccionario bilingüe** | **Sí** | En train casi no hay reseñas mezcladas, pero en eval sí. Se reemplaza una fracción de las palabras de una reseña en español por su equivalente en inglés (y al revés para reseñas en inglés) con los diccionarios MUSE. La estructura de la frase (orden, negación, conectores) no cambia, así que **la etiqueta se conserva**. |
| **B. Inserción de oraciones neutras de otro idioma** | **Sí** | Las oraciones de las reseñas neutrales son descriptivas (envío, tamaño, garantía) y casi nunca tienen palabras de opinión (lo verificamos abajo). Insertar 1-2 de ellas, **en otro idioma**, en posiciones intermedias de una reseña no cambia su veredicto, y produce reseñas multilingües a nivel de oración y más texto en en / pt / fr. Se inserta en el medio, nunca al inicio ni al final, porque ahí suele estar el veredicto. |
| Traducción automática con redes (MarianMT) | No | Es un modelo generativo: lo prohíbe el anuncio del curso. |
| Barajar el orden de las oraciones | No | **Cambia la etiqueta**: el sentimiento depende de qué opinión va al final o al inicio (lo vimos en la Parte 1). |
| Sinónimos / borrado / inserción aleatoria (EDA) | No | Puede borrar una negación o cambiar "excelente" por algo neutro, y así invertir el sentimiento sin cambiar la etiqueta (ruido). |

**Diccionarios MUSE** (Conneau et al., 2018, *Word Translation Without Parallel Data*): diccionarios bilingües de referencia publicados por Facebook Research con licencia **CC BY-NC 4.0** (uso académico no comercial permitido). Se usan `es-en` y `en-es` y se toma la primera traducción de cada palabra.

In [ ]:
import unicodedata, urllib.request

def sin_tildes(s):
    return ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))

def cargar_muse(par):
    """Descarga (una vez) y carga un diccionario MUSE como {palabra_origen: primera_traducción}."""
    ruta = DIR_AUG / f'muse_{par}.txt'
    if not ruta.exists():
        urllib.request.urlretrieve(f'https://dl.fbaipublicfiles.com/arrival/dictionaries/{par}.txt', ruta)
    dic = {}
    for linea in ruta.read_text(encoding='utf-8').splitlines():
        partes = linea.split()
        if len(partes) == 2 and partes[0] != partes[1]:
            dic.setdefault(partes[0], partes[1])
            dic.setdefault(sin_tildes(partes[0]), partes[1])   # muchas reseñas vienen sin tildes ("compre", "llego")
    return dic

MUSE = {'es': cargar_muse('es-en'), 'en': cargar_muse('en-es')}
print({k: f'{len(v):,} entradas' for k, v in MUSE.items()})

In [ ]:
def dividir_oraciones(texto):
    return [s for s in re.split(r'(?<=[.!?…])\s+', str(texto).strip()) if s]

TOKEN_SUST = re.compile(r"\w+|[^\w\s]+|\s+")
# Palabras que nunca se sustituyen: negaciones y conectores contrastivos, que definen el sentimiento
PROTEGIDAS = {'no', 'not', 'nunca', 'never', 'ni', 'nor', 'nada', 'nothing', 'pero', 'but', 'aunque', 'although',
              'sin', 'without', 'tampoco', 'neither', 'embargo', 'however', 'nadie', 'nobody'}

def code_switch_lexico(texto, dic, p, rng):
    """Reemplaza cada palabra traducible (>= 3 letras, no protegida) por su traducción con probabilidad p."""
    salida = []
    for tok in TOKEN_SUST.findall(texto):
        low = tok.lower()
        if len(low) >= 3 and low not in PROTEGIDAS and low in dic and rng.random() < p:
            trad = dic[low]
            salida.append(trad.capitalize() if tok[0].isupper() else trad)
        else:
            salida.append(tok)
    return ''.join(salida)

def insertar_neutras(texto, idioma, banco, rng, max_ins=2):
    """Inserta 1-2 oraciones neutras de OTRO idioma en posiciones intermedias (ni al inicio ni al final)."""
    oraciones = dividir_oraciones(texto)
    if len(oraciones) < 2:
        return None
    otros = [k for k in banco if k != idioma]
    for _ in range(rng.integers(1, max_ins + 1)):
        donante = banco[otros[rng.integers(len(otros))]]
        pos = rng.integers(1, len(oraciones))            # entre la primera y la última oración
        oraciones.insert(pos, donante[rng.integers(len(donante))])
    return ' '.join(oraciones)

#### 4.2 Verificación: ¿las oraciones de las reseñas neutrales realmente no tienen opinión?

La estrategia B solo es válida si las oraciones que se insertan no tienen sentimiento. Se mide qué fracción de ellas contiene alguna palabra de opinión frecuente (lista multilingüe). El banco de oraciones se construye **solo con reseñas neutrales de la partición de entrenamiento**, y se descartan las oraciones que contengan alguna palabra de opinión.

In [ ]:
OPINION_RE = re.compile(r'(excelente|buen[oa]|mal[oa]\b|deficiente|decepcion|encant|recomiend|pésim|pesim|horrible|genial|'
                        r'great|good|bad|terrible|love|hate|awful|amazing|disappoint|'
                        r'ótim|otim|bom\b|ruim|péssim|decepcion|'
                        r'parfait|nul\b|déçu|decu|génial|super\b)', re.I)

banco, fraccion_opinion = {}, {}
for idi in ['es', 'en', 'pt', 'fr']:
    ors = [s for t in tr.loc[(tr['label'] == 'neutral') & (tr['idioma'] == idi), 'text'] for s in dividir_oraciones(t)]
    con_op = [bool(OPINION_RE.search(s)) for s in ors]
    fraccion_opinion[idi] = np.mean(con_op)
    banco[idi] = [s for s, c in zip(ors, con_op) if not c and len(s.split()) >= 4]
display(pd.DataFrame({'oraciones en el banco': {k: len(v) for k, v in banco.items()},
                      '% con palabra de opinión (descartadas)': {k: round(100 * v, 2) for k, v in fraccion_opinion.items()}}))
for idi in banco:
    print(idi, '->', banco[idi][0])

#### 4.3 Construcción de los conjuntos aumentados

- **A (code-switching léxico):** a cada reseña de entrenamiento en español o inglés se le genera, con probabilidad `P_A`, una copia con el `P_SUST` = 30 % de sus palabras traducidas al otro idioma. La copia queda marcada como `es-en`.
- **B (inserción de neutras):** a una fracción `P_B` de **todas** las reseñas de entrenamiento (cualquier idioma) se le genera una copia con 1-2 oraciones neutras de otro idioma intercaladas.
- Ambas solo usan la partición de entrenamiento (`tr`), nunca la de validación ni `eval`.

In [ ]:
P_A, P_SUST, P_B = 0.3, 0.3, 0.4
rng = np.random.default_rng(SEMILLA)

filas_a, filas_b = [], []
for r in tr.itertuples():
    if r.idioma in ('es', 'en') and rng.random() < P_A:
        nuevo = code_switch_lexico(r.text, MUSE[r.idioma], P_SUST, rng)
        if nuevo != r.text:
            filas_a.append({'id': f'{r.id}_cs', 'text': nuevo, 'label': r.label, 'idioma': 'es-en', 'origen': 'A_code_switching'})
    if r.idioma in banco and rng.random() < P_B:
        nuevo = insertar_neutras(r.text, r.idioma, banco, rng)
        if nuevo:
            filas_b.append({'id': f'{r.id}_ins', 'text': nuevo, 'label': r.label, 'idioma': r.idioma, 'origen': 'B_insercion'})

aug_a, aug_b = pd.DataFrame(filas_a), pd.DataFrame(filas_b)
for a in (aug_a, aug_b):
    a['y'] = a['label'].map(C2I)
print(f'A, code-switching léxico: {len(aug_a):,} reseñas nuevas')
print(f'B, inserción de neutras:  {len(aug_b):,} reseñas nuevas')

ej = tr[tr['idioma'] == 'es'].iloc[3]
print('\nORIGINAL       :', ej['text'])
print('A (es->en 30 %):', code_switch_lexico(ej['text'], MUSE['es'], P_SUST, np.random.default_rng(1)))
print('B (inserción)  :', insertar_neutras(ej['text'], 'es', banco, np.random.default_rng(1)))
ej = tr[tr['idioma'] == 'en'].iloc[3]
print('\nORIGINAL       :', ej['text'])
print('A (en->es 30 %):', code_switch_lexico(ej['text'], MUSE['en'], P_SUST, np.random.default_rng(1)))

In [ ]:
tr['origen'] = 'original'
CONJUNTOS = {
    'sin_aug': tr,
    'A':       pd.concat([tr, aug_a], ignore_index=True),
    'A_B':     pd.concat([tr, aug_a, aug_b], ignore_index=True),
}
resumen = pd.DataFrame({k: v['idioma'].value_counts(normalize=True) * 100 for k, v in CONJUNTOS.items()}).reindex(IDIOMAS).fillna(0)
resumen['eval'] = P_EVAL.reindex(IDIOMAS).fillna(0) * 100
resumen.loc['n filas'] = [len(v) for v in CONJUNTOS.values()] + [len(eval_df)]
display(resumen.round(1).rename_axis('% por idioma'))

La aumentación A aumenta mucho el número de reseñas `es-en`, que en train eran apenas unas decenas. La B agrega mezclas de idiomas a nivel de oración en todos los idiomas, incluidos pt y fr.

### **5. Experimentos**

Se cambia **una cosa a la vez** para poder atribuir cada mejora a una decisión. Cada experimento genera un archivo de envío en `envios/`; así quedan los ≥ 5 envíos diferentes y cada uno corresponde a un paso documentado.

| Exp. | Modelo | Datos de entrenamiento | Qué se quiere medir |
|---|---|---|---|
| E1 | BiLSTM desde cero | sin aumentación | punto de partida secuencial sin preentrenamiento |
| E2 | BiLSTM desde cero | A + B | efecto de la aumentación sin preentrenamiento |
| E3 | XLM-R **base** (fine-tuning) | sin aumentación | efecto del preentrenamiento multilingüe |
| E4 | XLM-R base | + A (code-switching léxico) | efecto de la aumentación A |
| E5 | XLM-R base | + A + B (inserción de neutras) | efecto de la aumentación B |
| E6 | XLM-R **large** | mejor conjunto de E3-E5 | efecto del tamaño del *encoder* |
| E7 | XLM-R large, otra semilla | mejor conjunto de E3-E5 | variabilidad entre semillas |
| E8 | **Ensamble** E6 + E7 | (promedio de probabilidades) | efecto de combinar modelos |
| E9 | Ensamble E6 + E7 + mejor base | (promedio de probabilidades) | si un modelo distinto aporta diversidad |

La selección del conjunto de datos se hace con el modelo *base* (más barato) y solo la mejor configuración se entrena con el modelo *large*.

#### 5.1 Utilidades de evaluación

In [ ]:
RESULTADOS_CSV = DIR_RES / 'resultados.csv'
resultados = pd.read_csv(RESULTADOS_CSV).to_dict('records') if (RESULTADOS_CSV.exists() and REUTILIZAR_CACHE) else []

def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def acc_por_idioma(df, pred):
    ok = (df['y'].values == pred)
    return pd.Series(ok, index=df.index).groupby(df['idioma']).mean()

def registrar(nombre, descripcion, p_val, p_eval, segundos):
    """Guarda probabilidades, calcula métricas y escribe el archivo de envío del experimento."""
    np.save(DIR_RES / f'{nombre}_val.npy', p_val)
    np.save(DIR_RES / f'{nombre}_eval.npy', p_eval)
    pred = p_val.argmax(1)
    por_idi = acc_por_idioma(va, pred)
    fila = {
        'experimento': nombre, 'descripcion': descripcion,
        'acc_val': accuracy_score(va['y'], pred),
        'acc_ponderada_eval': float((por_idi * P_EVAL).dropna().sum() / P_EVAL[por_idi.index].sum()),
        **{f'acc_{k}': v for k, v in por_idi.items()},
        'minutos': segundos / 60,
    }
    global resultados
    resultados = [r for r in resultados if r['experimento'] != nombre] + [fila]
    pd.DataFrame(resultados).to_csv(RESULTADOS_CSV, index=False)
    envio = pd.DataFrame({'id': eval_df['id'], 'answer': [CLASES[i] for i in p_eval.argmax(1)]})
    assert len(envio) == len(sample) and envio['id'].equals(sample['id'])
    envio.to_csv(DIR_ENV / f'{nombre}.csv', index=False)
    print(f"{nombre}: acc_val={fila['acc_val']:.4f} | ponderada_eval={fila['acc_ponderada_eval']:.4f} | "
          + ' '.join(f'{k}={v:.3f}' for k, v in por_idi.items()))
    return fila

def ya_calculado(nombre):
    if nombre[:2] in ('E6', 'E7') and DIR_ENTREGA != DIR_MOD:
        for f in DIR_ENTREGA.glob(f'{nombre}.*'):             # recupera los pesos guardados si la sesión se reinició
            if not (DIR_MOD / f.name).exists():
                shutil.copy(f, DIR_MOD / f.name)
    return REUTILIZAR_CACHE and (DIR_RES / f'{nombre}_val.npy').exists() and any(r['experimento'] == nombre for r in resultados) \
        and (nombre[:2] in ('E1', 'E2', 'E3', 'E4', 'E5', 'E8', 'E9') or (DIR_MOD / f'{nombre}.weights.h5').exists())

def cargar_probs(nombre):
    return np.load(DIR_RES / f'{nombre}_val.npy'), np.load(DIR_RES / f'{nombre}_eval.npy')

def acc_de(nombre):
    return next((r['acc_val'] for r in resultados if r['experimento'] == nombre), float('nan'))

#### 5.2 Líneas de comparación: modelo clásico y BiLSTM desde cero (E1, E2)

**Referencia clásica (no es deep learning):** TF-IDF de palabras (1-2 gramas) y de caracteres (2-5 gramas) + regresión logística, el tipo de modelo de la Parte 1. No se envía a Kaggle ni cuenta como experimento: solo sirve para medir cuánto aportan las redes sobre los mismos datos y la misma validación.

**BiLSTM desde cero.** Arquitectura: `TextVectorization` (vocabulario aprendido de train) → `Embedding` 128 → **BiLSTM** 96 por dirección → *global max pooling* → `Dense` 64 → 3 *logits*.

A diferencia de la bolsa de palabras, la BiLSTM **procesa la secuencia en orden**, así que puede capturar dónde está cada opinión y el alcance de una negación. Pero no tiene conocimiento previo del idioma: todo lo aprende de ~18 mil reseñas, y en idiomas con pocos ejemplos (fr, pt) eso es poco. Sirve para medir cuánto aporta el preentrenamiento.

In [ ]:
# Referencia clásica, como los modelos de la Parte 1 (no es deep learning): TF-IDF de palabras y caracteres + regresión logística
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, make_union

ref_clasica = make_pipeline(
    make_union(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
               TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=3, sublinear_tf=True)),
    LogisticRegression(C=4, max_iter=2000))
ref_clasica.fit(tr['text'], tr['y'])
pred_ref = ref_clasica.predict(va['text'])
ACC_TFIDF = accuracy_score(va['y'], pred_ref)
print(f'TF-IDF + regresión logística: acc_val={ACC_TFIDF:.4f} | '
      + ' '.join(f'{k}={v:.3f}' for k, v in acc_por_idioma(va, pred_ref).items()))
del ref_clasica

In [ ]:
VOCAB, LARGO_PAL = 40000, 160

def construir_bilstm(textos_entrenamiento):
    vec = keras.layers.TextVectorization(max_tokens=VOCAB, output_sequence_length=LARGO_PAL)
    vec.adapt(tf.data.Dataset.from_tensor_slices(textos_entrenamiento).batch(512))
    ent = keras.Input(shape=(1,), dtype='string')
    x = vec(ent)
    x = keras.layers.Embedding(VOCAB, 128)(x)
    x = keras.layers.SpatialDropout1D(0.2)(x)
    x = keras.layers.Bidirectional(keras.layers.LSTM(96, return_sequences=True))(x)
    x = keras.layers.GlobalMaxPooling1D()(x)
    x = keras.layers.Dropout(0.3)(x)
    x = keras.layers.Dense(64, activation='relu')(x)
    sal = keras.layers.Dense(3, dtype='float32')(x)          # logits en float32 (estable con precisión mixta)
    modelo = keras.Model(ent, sal)
    modelo.compile(optimizer=keras.optimizers.Adam(2e-3),
                   loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
    return modelo

def experimento_bilstm(nombre, descripcion, df_entrenamiento, epocas=10):
    if ya_calculado(nombre):
        print(nombre, 'ya calculado: se reutiliza'); return
    keras.utils.set_random_seed(SEMILLA)
    t0 = time.time()
    X = textos_tf(df_entrenamiento['text'])
    modelo = construir_bilstm(X)
    pesos = str(DIR_MOD / f'{nombre}.weights.h5')
    hist = modelo.fit(X[:, None], df_entrenamiento['y'].values,
                      validation_data=(textos_tf(va['text'])[:, None], va['y'].values),
                      epochs=epocas, batch_size=128, verbose=2,
                      callbacks=[keras.callbacks.ModelCheckpoint(pesos, monitor='val_accuracy', mode='max',
                                                                 save_best_only=True, save_weights_only=True),
                                 keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=3)])
    modelo.load_weights(pesos)
    p_val = softmax(np.asarray(modelo.predict(textos_tf(va['text'])[:, None], batch_size=512, verbose=0), dtype='float32'))
    p_eval = softmax(np.asarray(modelo.predict(textos_tf(eval_df['text'])[:, None], batch_size=512, verbose=0), dtype='float32'))
    registrar(nombre, descripcion, p_val, p_eval, time.time() - t0)
    pd.DataFrame(hist.history).to_csv(DIR_RES / f'{nombre}_historia.csv', index=False)
    del modelo; keras.backend.clear_session(); gc.collect()

experimento_bilstm('E1_bilstm_sin_aug', 'BiLSTM desde cero, sin aumentación', CONJUNTOS['sin_aug'])
experimento_bilstm('E2_bilstm_aug', 'BiLSTM desde cero, + A + B', CONJUNTOS['A_B'])

#### 5.3 Transferencia de aprendizaje: XLM-RoBERTa (E3-E7)

**Modelo:** XLM-RoBERTa (Conneau et al., 2020), un *encoder-only* preentrenado con *masked language modeling* sobre 2.5 TB de texto CommonCrawl en 100 idiomas. Se carga desde **KerasHub** (licencia MIT) en dos tamaños:

| *Preset* | Capas | Dimensión | Parámetros |
|---|---|---|---|
| `xlm_roberta_base_multi` | 12 | 768 | ≈ 277 M |
| `xlm_roberta_large_multi` | 24 | 1024 | ≈ 558 M |

`TextClassifier` agrega sobre el *encoder* una cabeza de clasificación (capa densa + *dropout* + 3 *logits*) que usa la representación del token `<s>`. En el modelo *base* se ajusta (*fine-tuning*) **todo** el modelo; en el *large*, todo menos la matriz de *embeddings* de tokens (ver hiperparámetros). Esto corresponde a lo que permite el curso: embeddings contextuales de un *encoder-only* más una cabeza de clasificación.

**Por qué este modelo:**
- Un mismo *encoder* para los 4 idiomas y para el texto mezclado, con un vocabulario SentencePiece compartido. Lo aprendido en español se transfiere a portugués o francés.
- La autoatención mira la reseña completa a la vez, así que puede relacionar una concesión al inicio con el veredicto, o "no diría que" con el adjetivo que viene varias palabras después. El error de los modelos clásicos está casi todo en negativo ↔ positivo, que depende justo de eso.
- **Frente a mBERT**, el otro *encoder* multilingüe habitual, XLM-R se entrenó con unas 100 veces más texto por idioma (CommonCrawl en lugar de Wikipedia), con un vocabulario de más del doble (250k tokens SentencePiece frente a 110k WordPiece) y durante más tiempo. Conneau et al. (2020) muestran que mBERT estaba subentrenado y que estos cambios mejoran mucho la clasificación multilingüe:

 | XNLI, ajustado solo en inglés (exactitud %) | mBERT | XLM-R *base* | XLM-R *large* |
  |---|---|---|---|
  | Español | 74.3 | 80.7 | 85.1 |
  | Francés | 73.8 | 79.7 | 84.1 |
  | Inglés | 82.1 | 85.8 | 89.1 |
  | Media de 15 idiomas | 66.3 | 76.2 | 80.9 |

  La comparación justa con nuestro *base* es la columna del medio: tiene las mismas capas (12) y la misma dimensión (768) que mBERT, y aun así gana unos 6 puntos en español y francés. Parte de la ventaja del *large* se debe solo a su tamaño. XNLI es una tarea de inferencia (con 3 clases, como la nuestra), no de sentimiento, y no incluye portugués. Por eso estas cifras indican la calidad de las representaciones, no el resultado esperado con nuestras reseñas.
- El tamaño *large* tiene más capacidad para los casos de ironía y negación larga. Se entrena solo con la mejor configuración de datos porque cuesta ~3 veces más.

**Hiperparámetros** (valores estándar para *fine-tuning* de modelos tipo BERT, ajustados a la GPU T4 de Colab):
- AdamW con *warmup* lineal del 10 % y decaimiento coseno, *weight decay* 0.01, *clipping* del gradiente a norma 1.
- *base*: lr 2e-5, lote 32, 3 épocas.
- *large*: lr 1e-5 (más bajo porque los modelos grandes son más inestables), lote 8, **1 época** y la matriz de *embeddings* de tokens **congelada**. Es la configuración que cupo en los 16 GB de la T4: esa matriz tiene ≈ 250,000 × 1,024 ≈ 256 M parámetros, casi la mitad del modelo, y congelarla ahorra ≈ 3 GB de gradientes y estados de AdamW. Se pierde poco, porque esos *embeddings* ya vienen entrenados en 100 idiomas y las 24 capas de atención sí se ajustan. Se usa 1 época porque cada época del *large* tarda 1-2 horas en la T4 y la sesión de Colab tiene un tiempo limitado; con lote 8 son ≈ 2,300 actualizaciones de pesos, y con eso el *large* ya iguala en validación al *base* entrenado 3 épocas.
- Longitud máxima `MAX_LEN` tokens, calculada abajo para cubrir ~99.5 % de las reseñas **sin truncarlas** (resultó 160): el veredicto puede estar al inicio o al final, así que no conviene cortar ninguno de los dos extremos.
- Se guarda el **mejor checkpoint según la exactitud de validación** (en el *large*, con 1 época, es el de esa época).
- **Falta de memoria:** si la GPU se queda sin memoria, se reintenta automáticamente con la mitad del lote.
- **Control de colapso:** a veces el *fine-tuning* de un modelo grande no arranca y se queda prediciendo una sola clase. Si después de la primera época la exactitud de validación es menor a 0.5, se detiene y se reintenta con otra semilla y un lr 30 % menor. Le pasó al primer intento de E6 (exactitud 0.37): el segundo intento (semilla 43, lr 7e-6) entrenó bien.

In [ ]:
PRESET_BASE, PRESET_LARGE = 'xlm_roberta_base_multi', 'xlm_roberta_large_multi'
tokenizador = keras_hub.models.TextClassifierPreprocessor.from_preset(PRESET_BASE).tokenizer
ids = tokenizador(list(pd.concat([train['text'], eval_df['text']]).sample(4000, random_state=SEMILLA)))
largos = ids.row_lengths().numpy() if hasattr(ids, 'row_lengths') else np.array([len(x) for x in ids])
MAX_LEN = int(min(256, math.ceil((np.percentile(largos, 99.5) + 2) / 32) * 32))
print(f'Tokens por reseña: media={largos.mean():.0f}, p95={np.percentile(largos, 95):.0f}, máx={largos.max()} -> MAX_LEN={MAX_LEN}')

In [ ]:
def construir_transformer(preset, lr, pasos_totales, max_len=None, congelar_embeddings=False):
    max_len = max_len or MAX_LEN
    pre = keras_hub.models.TextClassifierPreprocessor.from_preset(preset, sequence_length=max_len)
    clf = keras_hub.models.TextClassifier.from_preset(preset, num_classes=3, preprocessor=pre)
    if congelar_embeddings:
        clf.backbone.token_embedding.trainable = False   # ahorra ~3 GB de memoria de GPU en el modelo large
    calentamiento = max(1, int(0.1 * pasos_totales))
    horario = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.0, warmup_target=lr, warmup_steps=calentamiento,
        decay_steps=max(1, pasos_totales - calentamiento))
    opt = keras.optimizers.AdamW(learning_rate=horario, weight_decay=0.01, global_clipnorm=1.0)
    clf.compile(optimizer=opt, loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                metrics=['accuracy'], jit_compile=not congelar_embeddings)
    return clf

class DetenerSiColapsa(keras.callbacks.Callback):
    """Detiene el entrenamiento si tras la primera época el modelo no aprendió (predice casi al azar)."""
    def on_epoch_end(self, epoch, logs=None):
        if epoch == 0 and (logs or {}).get('val_accuracy', 1.0) < 0.5:
            print('  -> colapso detectado: se detiene este intento')
            self.model.stop_training = True

def predecir_probs(modelo, textos, lote=64):
    return softmax(np.asarray(modelo.predict(textos_tf(textos), batch_size=lote, verbose=0), dtype='float32'))

def experimento_transformer(nombre, descripcion, df_entrenamiento, preset=PRESET_BASE, lr=2e-5,
                            epocas=3, lote=32, max_len=None, semilla=SEMILLA, congelar_embeddings=False):
    if ya_calculado(nombre):
        print(nombre, 'ya calculado: se reutiliza'); return
    t0 = time.time()
    pesos = str(DIR_MOD / f'{nombre}.weights.h5')
    X, y = textos_tf(df_entrenamiento['text']), df_entrenamiento['y'].values
    Xv, yv = textos_tf(va['text']), va['y'].values
    lote_i, intento, hist = lote, 0, None
    while intento < 3:
        keras.backend.clear_session(); gc.collect()
        keras.utils.set_random_seed(semilla + intento)
        lr_i = lr * (0.7 ** intento)
        try:
            clf = construir_transformer(preset, lr_i, math.ceil(int(X.shape[0]) / lote_i) * epocas, max_len, congelar_embeddings)
            hist = clf.fit(x=X, y=y, validation_data=(Xv, yv), batch_size=lote_i, epochs=epocas, verbose=1,
                           callbacks=[keras.callbacks.ModelCheckpoint(pesos, monitor='val_accuracy', mode='max',
                                                                      save_best_only=True, save_weights_only=True),
                                      DetenerSiColapsa()])
        except Exception as e:
            if ('RESOURCE_EXHAUSTED' in str(e) or 'OOM' in str(e)) and lote_i > 1:
                lote_i //= 2                       # sin memoria de GPU: se reintenta con un lote más pequeño
                print(f'  -> sin memoria de GPU: se reintenta con lote {lote_i}')
                continue
            raise
        if max(hist.history['val_accuracy']) >= 0.5 or intento == 2:
            break
        intento += 1
        del clf
    clf.load_weights(pesos)
    p_val, p_eval = predecir_probs(clf, Xv), predecir_probs(clf, eval_df['text'].values)
    registrar(nombre, descripcion, p_val, p_eval, time.time() - t0)
    pd.DataFrame(hist.history).to_csv(DIR_RES / f'{nombre}_historia.csv', index=False)
    json.dump({'preset': preset, 'lr': lr_i, 'epocas': epocas, 'lote': lote_i, 'max_len': max_len or MAX_LEN,
               'semilla': semilla + intento, 'congelar_embeddings': congelar_embeddings}, open(DIR_MOD / f'{nombre}.json', 'w'))
    del clf; keras.backend.clear_session(); gc.collect()
    try:
        __import__('ctypes').CDLL('libc.so.6').malloc_trim(0)     # devuelve memoria RAM al sistema (solo Linux)
    except Exception:
        pass


**E3-E5: XLM-R base, con y sin aumentación**

In [ ]:
experimento_transformer('E3_xlmr_sin_aug', 'XLM-R base, sin aumentación', CONJUNTOS['sin_aug'])

In [ ]:
experimento_transformer('E4_xlmr_A', 'XLM-R base, + A (code-switching léxico)', CONJUNTOS['A'])

In [ ]:
experimento_transformer('E5_xlmr_A_B', 'XLM-R base, + A + B (inserción de neutras)', CONJUNTOS['A_B'])

**E6-E7: XLM-R large con el mejor conjunto de datos**

El conjunto de datos se elige con la exactitud de validación de E3-E5. Ganó **sin aumentación** (0.9086) frente a A (0.9072) y A + B (0.9062). Las diferencias son menores que el ruido (McNemar p > 0.6, sección 6), así que ninguna aumentación justifica su costo: agrega reseñas y alarga cada época (E5 tardó 54 min frente a 37 de E3), algo que en el *large* se multiplica.

Se entrenan dos modelos *large* que solo se diferencian en la semilla (inicialización de la cabeza y orden de los lotes). Sirven para dos cosas: medir cuánto varía el resultado por azar y tener dos modelos para el ensamble.

In [ ]:
EXP_DE = {'sin_aug': 'E3_xlmr_sin_aug', 'A': 'E4_xlmr_A', 'A_B': 'E5_xlmr_A_B'}
MEJOR_DATOS = max(EXP_DE, key=lambda k: acc_de(EXP_DE[k]) if not np.isnan(acc_de(EXP_DE[k])) else -1)
MEJOR_BASE = EXP_DE[MEJOR_DATOS]
print({k: round(acc_de(v), 4) for k, v in EXP_DE.items()}, '-> mejor conjunto de datos:', MEJOR_DATOS)

In [ ]:
experimento_transformer('E6_xlmr_large', f'XLM-R large, datos {MEJOR_DATOS}', CONJUNTOS[MEJOR_DATOS],
                        preset=PRESET_LARGE, lr=1e-5, lote=8, epocas=1, semilla=SEMILLA, congelar_embeddings=True)
for f in DIR_MOD.glob('E6_xlmr_large.*'):
    if DIR_ENTREGA != DIR_MOD:
        shutil.copy(f, DIR_ENTREGA / f.name)   # copia a Drive para no perder los pesos


In [ ]:
experimento_transformer('E7_xlmr_large_s2', f'XLM-R large, datos {MEJOR_DATOS}, semilla 2', CONJUNTOS[MEJOR_DATOS],
                        preset=PRESET_LARGE, lr=1e-5, lote=8, epocas=1, semilla=2026, congelar_embeddings=True)
for f in DIR_MOD.glob('E7_xlmr_large_s2.*'):
    if DIR_ENTREGA != DIR_MOD:
        shutil.copy(f, DIR_ENTREGA / f.name)   # copia a Drive para no perder los pesos


#### 5.4 Ensamble (E8, E9)

Dos modelos entrenados con semillas distintas se equivocan en reseñas parcialmente distintas. Al **promediar sus probabilidades**, los errores que no comparten tienden a cancelarse: es la misma idea del *bagging* y del *stacking* de la Parte 1, aplicada a redes. No se entrena nada nuevo: solo se promedian las probabilidades ya guardadas de cada modelo.

- **E8:** los dos XLM-R large.
- **E9:** los dos large más el mejor XLM-R base, para ver si un modelo de otro tamaño aporta diversidad o solo baja el promedio.

In [ ]:
MIEMBROS_ENSAMBLE = {}

def ensamble(nombre, descripcion, miembros):
    miembros = [m for m in miembros if (DIR_RES / f'{m}_val.npy').exists()]
    if len(miembros) < 2:
        print(nombre, ': faltan miembros, se omite'); return
    p_val = np.mean([cargar_probs(m)[0] for m in miembros], axis=0)
    p_eval = np.mean([cargar_probs(m)[1] for m in miembros], axis=0)
    MIEMBROS_ENSAMBLE[nombre] = miembros
    registrar(nombre, descripcion, p_val, p_eval, 0)

ensamble('E8_ensamble_large', 'Promedio de E6 + E7', ['E6_xlmr_large', 'E7_xlmr_large_s2'])
ensamble('E9_ensamble_large_base', f'Promedio de E6 + E7 + {MEJOR_BASE}', ['E6_xlmr_large', 'E7_xlmr_large_s2', MEJOR_BASE])

### **6. Comparación de experimentos y efecto de la aumentación**

In [ ]:
tabla_res = pd.DataFrame(resultados).sort_values('experimento').set_index('experimento')
cols_idi = [f'acc_{i}' for i in IDIOMAS if f'acc_{i}' in tabla_res]
display(tabla_res[['descripcion', 'acc_val', 'acc_ponderada_eval', *cols_idi, 'minutos']]
        .style.format({c: '{:.4f}' for c in ['acc_val', 'acc_ponderada_eval', *cols_idi]} | {'minutos': '{:.1f}'})
        .background_gradient(subset=['acc_val'], cmap='Greens'))

ax = tabla_res[cols_idi].rename(columns=lambda c: c[4:]).plot.bar(figsize=(15, 4), rot=30)
ax.set_ylim(max(0, tabla_res[cols_idi].min().min() - 0.05), 1)
ax.set_title('Exactitud de validación por idioma y experimento'); ax.legend(ncol=5)
plt.tight_layout(); plt.show()

**¿Las diferencias son reales o ruido?** Se usa la **prueba de McNemar** sobre las mismas reseñas de validación. Solo cuentan las reseñas en las que los dos modelos discrepan; si el cambio no tuviera efecto, cada modelo acertaría la mitad de esas discrepancias. La comparación E6 vs E7 (misma configuración, distinta semilla) muestra cuánta diferencia produce solo el azar.

In [ ]:
def mcnemar(a, b):
    pa, pb = cargar_probs(a)[0].argmax(1), cargar_probs(b)[0].argmax(1)
    ok_a, ok_b = pa == va['y'].values, pb == va['y'].values
    solo_a, solo_b = int((ok_a & ~ok_b).sum()), int((~ok_a & ok_b).sum())
    p = binomtest(solo_b, solo_a + solo_b, 0.5).pvalue if solo_a + solo_b else 1.0
    print(f'{b} vs {a}: Δacc = {ok_b.mean() - ok_a.mean():+.4f} | solo acierta {a}: {solo_a}, '
          f'solo acierta {b}: {solo_b} | p-valor McNemar = {p:.4f}')

hechos = set(tabla_res.index)
print('--- Con vs sin aumentación ---')
for a, b in [('E1_bilstm_sin_aug', 'E2_bilstm_aug'), ('E3_xlmr_sin_aug', 'E4_xlmr_A'),
             ('E4_xlmr_A', 'E5_xlmr_A_B'), ('E3_xlmr_sin_aug', 'E5_xlmr_A_B')]:
    if {a, b} <= hechos:
        mcnemar(a, b)
print('--- Preentrenamiento, tamaño, semilla y ensamble ---')
for a, b in [('E1_bilstm_sin_aug', 'E3_xlmr_sin_aug'), (MEJOR_BASE, 'E6_xlmr_large'),
             ('E6_xlmr_large', 'E7_xlmr_large_s2'), ('E6_xlmr_large', 'E8_ensamble_large'),
             ('E8_ensamble_large', 'E9_ensamble_large_base')]:
    if {a, b} <= hechos:
        mcnemar(a, b)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for n in tabla_res.index:
    h = DIR_RES / f'{n}_historia.csv'
    if h.exists():
        hh = pd.read_csv(h)
        axes[0].plot(range(1, len(hh) + 1), hh['val_accuracy'], marker='o', label=n)
        axes[1].plot(range(1, len(hh) + 1), hh['loss'], marker='o', label=n)
axes[0].set_title('Exactitud de validación por época'); axes[1].set_title('Pérdida de entrenamiento por época')
for a in axes: a.set_xlabel('época'); a.legend(fontsize=7)
plt.tight_layout(); plt.show()

**Análisis** (exactitud de validación sobre 2,047 reseñas; el p-valor es de McNemar)

- **Modelo clásico → BiLSTM:** TF-IDF + regresión logística llega a 0.738 y la BiLSTM desde cero (E1) a 0.836, +9.8 puntos. Leer la secuencia en orden ya ayuda mucho con las negaciones y los contrastes, que una bolsa de palabras no distingue.
- **Preentrenamiento (E1 → E3):** es la mejora más grande entre las redes: de 0.836 (BiLSTM) a 0.909 (XLM-R base), +7.3 puntos, con p < 0.001 (XLM-R acierta 234 reseñas que la BiLSTM falla, y la BiLSTM solo 85 que XLM-R falla). Sube en todos los idiomas, pero sobre todo en los que tienen menos datos: fr pasa de 0.732 a 0.909, pt de 0.772 a 0.897 y en de 0.715 a 0.860 (es sube de 0.883 a 0.921).
- **Code-switching léxico (E3 → E4):** no hay efecto: 0.9086 → 0.9072 (p = 0.83) y la exactitud ponderada a eval casi no cambia (0.9055 → 0.9053). Por idioma sube un poco en en (+1.3 puntos), pt (+0.9) y fr (+0.5), pero baja en es (−0.7), que es la gran mayoría de la validación. En `es-en` solo hay 6 reseñas de validación, así que no se puede medir el efecto que se buscaba.
- **Inserción de neutras (E4 → E5):** tampoco ayuda: 0.9072 → 0.9062 (p = 0.93). Frente a E3, la diferencia es −0.24 puntos (p = 0.69).
- **Aumentación en la BiLSTM (E1 → E2):** sube +0.7 puntos (0.836 → 0.843), un poco más que en XLM-R. Es lo esperado: la BiLSTM depende más de la cantidad de datos, mientras que XLM-R ya trae conocimiento de los cuatro idiomas y del texto mezclado desde el preentrenamiento. Aun así no es significativa (p = 0.36).
- **Tamaño del modelo (base → large, E6):** igual en validación (0.9086 en los dos, p = 1.0). Large mejora en fr (+1.9 puntos) y empeora un poco en es (−0.4). Large se entrenó con 1 época y los *embeddings* de tokens congelados para que cupiera en la GPU T4; es probable que así no aproveche toda su capacidad.
- **Semilla (E6 vs E7):** misma configuración, distinta semilla, y la diferencia es −0.9 puntos (p = 0.13); los dos modelos coinciden en el 93.7 % de las reseñas de validación y en el 93.5 % de las de eval. Ese ≈ 1 punto es el ruido contra el que hay que comparar todas las demás mejoras: ninguna diferencia entre E3, E4, E5 y E6 lo supera.
- **Ensamble (E8, E9):** E8 (0.9096) gana +0.1 puntos sobre E6 (p = 0.89) y +1.0 sobre E7, esta vez con una diferencia significativa (p = 0.03). Es decir, el ensamble no supera claramente al mejor modelo individual, pero elimina el riesgo de quedarse con una semilla mala: con un solo *large* podríamos haber entregado E7. Agregar el modelo base (E9, 0.9106) sube otros +0.1 puntos (p = 0.89), que es ruido.
- **Ranking de validación vs. score público de Kaggle:** el orden no es idéntico. En validación E9 > E8 > E3 ≈ E6 > E4 > E5 > E7, y en Kaggle E8 (0.8939) > E9 (0.8919) > E6 (0.8915) > E7 (0.8865) > E5 (0.8762). Coinciden en que E8 y E9 son los mejores, y E5 (con las dos aumentaciones) queda por debajo de los demás transformers enviados (E3 y E4 no se enviaron). El score de Kaggle queda entre 1.1 (E7) y 2.8 (E5) puntos por debajo de la exactitud ponderada a eval (1.3 en E8). Es lo que anticipaba la exploración: eval tiene casi el doble de negaciones largas y de consecuencias que train, una dificultad que la ponderación por idioma no captura.

### **7. Análisis de errores por idioma**

Sobre el modelo final (ensamble E8, ver sección 8) se analiza la exactitud y la matriz de confusión por idioma, la exactitud por tipo de reseña difícil y ejemplos concretos de errores.

In [ ]:
# Se analiza el modelo final (E8). E9 tiene 0.1 puntos más de exactitud de validación, pero la diferencia es ruido (McNemar p = 0.89)
MEJOR = 'E8_ensamble_large'
p_val_mejor = cargar_probs(MEJOR)[0]
va_err = va.assign(pred=p_val_mejor.argmax(1), conf=p_val_mejor.max(1))
va_err['acierto'] = va_err['pred'] == va_err['y']
print('Mejor experimento:', MEJOR)
print(classification_report(va_err['y'], va_err['pred'], target_names=CLASES, digits=4))

por_idioma = va_err.groupby('idioma').agg(n=('acierto', 'size'), exactitud=('acierto', 'mean')).reindex(IDIOMAS)
por_idioma['errores'] = (por_idioma['n'] * (1 - por_idioma['exactitud'])).round().astype('Int64')
display(por_idioma.style.format({'exactitud': '{:.4f}'}))

fig, axes = plt.subplots(1, len(IDIOMAS), figsize=(4 * len(IDIOMAS), 3.6))
for ax, idi in zip(axes, IDIOMAS):
    sub = va_err[va_err['idioma'] == idi]
    if len(sub):
        ConfusionMatrixDisplay(confusion_matrix(sub['y'], sub['pred'], labels=[0, 1, 2]),
                               display_labels=['neg', 'neu', 'pos']).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{idi} (n={len(sub)}, acc={sub["acierto"].mean():.3f})')
plt.tight_layout(); plt.show()

In [ ]:
pat_va = marcar_patrones(va_err)
tipo = pd.DataFrame({
    'n': pat_va.sum(),
    'exactitud': [va_err.loc[pat_va[c], 'acierto'].mean() for c in pat_va],
}).sort_values('exactitud')
tipo.loc['(sin ningún patrón)'] = [int((~pat_va.any(axis=1)).sum()), va_err.loc[~pat_va.any(axis=1), 'acierto'].mean()]
display(tipo.style.format({'exactitud': '{:.4f}'}).set_caption('Exactitud por tipo de reseña'))

errores = va_err[~va_err['acierto']].copy()
errores['real'] = errores['y'].map(dict(enumerate(CLASES)))
errores['predicho'] = errores['pred'].map(dict(enumerate(CLASES)))
print('Tipos de confusión más frecuentes:')
display(errores.groupby(['real', 'predicho']).size().sort_values(ascending=False).to_frame('n'))

# ¿El modelo sabe cuándo duda? Exactitud según la confianza de la predicción
va_err['tramo_conf'] = pd.cut(va_err['conf'], [0, 0.6, 0.8, 0.95, 1.0])
display(va_err.groupby('tramo_conf', observed=True).agg(n=('acierto', 'size'), exactitud=('acierto', 'mean')).round(4))

In [ ]:
for idi in IDIOMAS:
    sub = errores[errores['idioma'] == idi].sort_values('conf', ascending=False).head(3)
    if len(sub):
        print(f'\n=========== {idi}: errores con mayor confianza ===========')
        for r in sub.itertuples():
            print(f'[real={r.real} | pred={r.predicho} | conf={r.conf:.2f}]\n  {r.text}\n')

**Análisis de errores** (modelo final, E8, sobre la validación: 185 errores de 2,047 reseñas)

- **Idioma:** el que más falla es el inglés (0.855), seguido de portugués (0.893), español (0.919) y francés (0.943). Inglés y portugués tienen pocas reseñas en validación (235 y 224), así que las diferencias entre idiomas tienen bastante incertidumbre (≈ ±2 puntos de error estándar en esos idiomas). `es-en` tiene solo 6 reseñas en validación (acierta 2), por lo que no permite sacar conclusiones.
- **Confusión dominante:** todo el error es negativo ↔ positivo, igual que en la Parte 1, y la clase neutral está resuelta (precisión y recall de 1.00). 133 errores son reseñas negativas predichas como positivas y 52 positivas predichas como negativas, así que el modelo se inclina hacia positivo (recall de negativo 0.82, precisión de positivo 0.84). Es coherente con los tipos de reseña difíciles: una reseña negativa escrita con ironía, con una concesión inicial ("la pantalla es bonita, pero…") o que solo describe una consecuencia contiene palabras positivas que el modelo sobrevalora.
- **Tipo de reseña difícil:** las comparaciones ("esperaba…", "mejor que…") tienen 0.825 de exactitud (137 reseñas) y las de consecuencia (devoluciones, reembolsos) 0.845 (71 reseñas). Las ironías son las más difíciles (0.75), pero solo hay 8. Las reseñas con negación larga salen bien (0.983 en 57) y las que no tienen ninguno de los patrones marcados llegan a 0.915.
- **Etiquetas ambiguas o ruidosas:** el modelo sabe cuándo duda. Cuando la confianza es mayor a 0.95 (el 82 % de las reseñas) acierta el 99.0 %, y cuando es menor a 0.8 (324 reseñas) acierta el 51 %, casi como lanzar una moneda. Solo hay 16 errores con confianza mayor a 0.95, y es ahí donde es más probable que la etiqueta misma esté mal o sea ambigua; ese error no se puede reducir con un mejor modelo.
- **Con más tiempo:** (1) más semillas de XLM-R large en el ensamble, (2) entrenar large con más épocas y sin congelar los *embeddings* (con una GPU de más memoria), (3) reentrenar con train + validación para el modelo final, y (4) revisar las reseñas con baja confianza para depurar etiquetas.

### **8. Modelo final, guardado y envío**

El modelo final es el ensamble **E8** (promedio de dos XLM-R large con semillas distintas). Es el envío con mejor *score* público en Kaggle (0.89390) y el segundo en validación (0.9096). E9 tiene una exactitud de validación 0.1 puntos mayor, pero esa diferencia es ruido (p = 0.89), en Kaggle quedó por debajo (0.89194) y necesitaría además el modelo base de E3, cuyos pesos no se conservaron (para ahorrar espacio, solo se guardaron en Drive los pesos de los dos *large*).

- El modelo final son los miembros del ensamble: se guarda un archivo `.keras` por miembro y la predicción es el promedio de sus probabilidades.
- Cada archivo `.keras` contiene el modelo **completo** (arquitectura + pesos + tokenizador), así que recibe texto crudo.
- Se verifica que cada archivo, cargado de nuevo, reproduce las predicciones del experimento.
- Al final se borran los pesos intermedios (`.weights.h5`) del disco local para no llenarlo.

In [ ]:
# Modelo final: ensamble E8 (E6 + E7, los dos XLM-R large). E9 tiene acc. de validación apenas mayor (0.9106 vs 0.9096), pero exige
# el modelo base de E3, cuyos pesos finales no se conservaron; la diferencia es ruido (ver McNemar). Cambiar a mano si Kaggle indica otro.
FINAL = 'E8_ensamble_large'
if FINAL not in MIEMBROS_ENSAMBLE and not FINAL.startswith(('E1', 'E2', 'E3', 'E4', 'E5', 'E6', 'E7')):
    raise RuntimeError(f'{FINAL} no está definido: ejecutar antes la celda del ensamble')
MIEMBROS = MIEMBROS_ENSAMBLE.get(FINAL, [FINAL])
print('Modelo final:', FINAL, '| miembros:', MIEMBROS)

probs_eval = []
for i, m in enumerate(MIEMBROS, 1):
    keras.backend.clear_session(); gc.collect()
    ruta = DIR_MOD / (f'modelo_final_{i}_{m}.keras' if len(MIEMBROS) > 1 else 'modelo_final.keras')
    if not ruta.exists():
        if m.startswith('E1') or m.startswith('E2'):
            conj = CONJUNTOS['sin_aug'] if m.startswith('E1') else CONJUNTOS['A_B']
            modelo = construir_bilstm(textos_tf(conj['text']))
        else:
            for f in DIR_ENTREGA.glob(f'{m}.*'):                      # en Colab, recupera los pesos guardados en Drive
                if not (DIR_MOD / f.name).exists():
                    shutil.copy(f, DIR_MOD / f.name)
            if not (DIR_MOD / f'{m}.json').exists() or not (DIR_MOD / f'{m}.weights.h5').exists():
                raise FileNotFoundError(f'Faltan los pesos de {m} en {DIR_MOD}: volver a ejecutar su experimento')
            cfg = json.load(open(DIR_MOD / f'{m}.json'))
            modelo = construir_transformer(cfg['preset'], cfg['lr'], 1000, cfg['max_len'], cfg.get('congelar_embeddings', False))
        modelo.load_weights(str(DIR_MOD / f'{m}.weights.h5'))
        modelo.save(ruta)
        del modelo; keras.backend.clear_session(); gc.collect()
    # Verificación de reproducibilidad: cargar desde el archivo y predecir eval
    recargado = keras.models.load_model(ruta, compile=False)
    X_eval = textos_tf(eval_df['text'])
    if m.startswith('E1') or m.startswith('E2'):
        X_eval = X_eval[:, None]
    p = softmax(np.asarray(recargado.predict(X_eval, batch_size=64, verbose=0), dtype='float32'))
    iguales = (p.argmax(1) == cargar_probs(m)[1].argmax(1)).mean()
    print(f'{ruta.name}: {ruta.stat().st_size / 1e6:.0f} MB | predicciones iguales a las del experimento: {iguales:.2%}')
    probs_eval.append(p)
    del recargado

p_eval_final = np.mean(probs_eval, axis=0)
envio_final = pd.DataFrame({'id': eval_df['id'], 'answer': [CLASES[i] for i in p_eval_final.argmax(1)]})
envio_final.to_csv(DIR_ENV / 'envio_final.csv', index=False)
print('Coincidencia de envio_final con el envío del experimento:',
      f"{(envio_final['answer'] == pd.read_csv(DIR_ENV / f'{FINAL}.csv')['answer']).mean():.2%}")
display(envio_final['answer'].value_counts(normalize=True).round(3).to_frame('proporción predicha en eval'))

if DIR_ENTREGA != DIR_MOD:
    for f in DIR_MOD.glob('modelo_final*.keras'):
        shutil.copy(f, DIR_ENTREGA / f.name)                         # el modelo final queda también en Drive

LIMPIAR_PESOS = True
if LIMPIAR_PESOS:
    for f in DIR_MOD.glob('*.weights.h5'):
        f.unlink()

**Cómo usar el modelo entregado** (no requiere este notebook):

```python
import glob, keras, keras_hub, numpy as np, tensorflow as tf
from scipy.special import softmax

textos = tf.constant(['Lo devolví a la semana.', 'Works great, love it!'])
archivos = sorted(glob.glob('modelo_final*.keras'))          # 1 archivo, o varios si es un ensamble
probs = np.mean([softmax(keras.models.load_model(a, compile=False).predict(textos).astype('float32'), axis=1)
                 for a in archivos], axis=0)
print(np.array(['negativo', 'neutral', 'positivo'])[probs.argmax(1)])
```

### **9. Conclusiones**

1. **Modelo final:** ensamble E8, el promedio de dos XLM-R large entrenados con semillas distintas (exactitud de validación 0.9096, ponderada a eval 0.9070, score público de Kaggle 0.89390, el mejor de nuestros envíos). E9 tiene una validación apenas mayor (0.9106, diferencia no significativa), pero en Kaggle quedó por debajo de E8 (0.89194) y además necesita el modelo base de E3, cuyos pesos no se conservaron.
2. **Lo que más aportó, ordenado por ganancia medida:** (a) el preentrenamiento multilingüe, +7.3 puntos sobre la BiLSTM (p < 0.001), de lejos lo más importante; (b) el ensamble, que no supera claramente al mejor *large* (+0.1, p = 0.89) pero sí al peor (+1.0, p = 0.03), y en Kaggle gana +0.25 puntos sobre el mejor individual (E6); (c) el tamaño del modelo no mejoró la validación (0.9086 en base y en large), aunque large se entrenó con 1 época y con los *embeddings* congelados; (d) la aumentación, sin efecto medible.
3. **Efecto de la aumentación:** sin efecto significativo. En XLM-R base, E3 → E4 da −0.15 puntos (p = 0.83) y E3 → E5 −0.24 puntos (p = 0.69), y en Kaggle E5 es el más bajo de los transformers enviados (0.87622). Por idioma, la aumentación A sube un poco en en, pt y fr, pero baja en es. El objetivo principal de A era el code-switching, que según nuestro detector es solo el 0.5 % de eval (33 reseñas) y tiene 6 reseñas en validación, así que su efecto no se podía medir y, aunque existiera, movería poco la exactitud global. En la BiLSTM la aumentación da +0.7 puntos, tampoco significativo (p = 0.36): ayuda más a un modelo sin preentrenamiento, porque XLM-R ya trae ese conocimiento multilingüe. Por eso E6 y E7 se entrenaron sin aumentación: la decisión se tomó con la evidencia de E3-E5.
4. **Errores restantes:** el 9 % de error es todo negativo ↔ positivo (72 % son negativas predichas como positivas), se concentra en comparaciones, consecuencias e ironía, y en inglés y portugués. Una parte parece ruido de etiquetas: con confianza mayor a 0.95 el modelo acierta el 99 % y sus 16 errores ahí son sospechosos.
5. **Comparación con la Parte 1:** TF-IDF + regresión logística 0.738 → BiLSTM desde cero 0.836 (0.843 con aumentación) → XLM-R base 0.909 → ensamble de dos large 0.910, todos sobre la misma validación. Cada paso aprovecha más contexto, y el mayor salto viene de pasar a modelos secuenciales y, sobre todo, del preentrenamiento.
6. **Limitaciones:** las diferencias de menos de ≈ 1 punto entre experimentos están dentro del ruido por semilla (E6 vs E7), y con una sola partición de validación no se puede afinar más. El entrenamiento de large se limitó por la memoria y el tiempo de la GPU de Colab (1 época, *embeddings* congelados). El primer intento de E6 colapsó (exactitud 0.37) y se reintentó automáticamente con otra semilla y menor tasa de aprendizaje; el resultado reportado es el del segundo intento. El detector de idioma es heurístico y probablemente subestima el code-switching.

### Registro de envíos a Kaggle

| # | Archivo (`envios/`) | Experimento | Acc. validación | Score público Kaggle |
|---|---|---|---|---|
| 1 | E1_bilstm_sin_aug.csv | BiLSTM sin aug | 0.8359 | no registrado |
| 2 | E2_bilstm_aug.csv | BiLSTM + A + B | 0.8432 | no registrado |
| 3 | E3_xlmr_sin_aug.csv | XLM-R base sin aug | 0.9086 | no registrado |
| 4 | E4_xlmr_A.csv | XLM-R base + A | 0.9072 | no registrado |
| 5 | E5_xlmr_A_B.csv | XLM-R base + A + B | 0.9062 | 0.87622 |
| 6 | E6_xlmr_large.csv | XLM-R large | 0.9086 | 0.89145 |
| 7 | E7_xlmr_large_s2.csv | XLM-R large, semilla 2 | 0.8999 | 0.88654 |
| 8 | E8_ensamble_large.csv | Ensamble E6 + E7 | 0.9096 | **0.89390** |
| 9 | E9_ensamble_large_base.csv | Ensamble E6 + E7 + base | 0.9106 | 0.89194 |
| 10 | envio_final.csv | modelo final (E8) | 0.9096 | igual al de E8 (mismas predicciones) |

Además, el grupo hizo cinco envíos anteriores con otros modelos (envio1 a envio5): 0.79076, 0.79911, 0.88654, 0.88654 y 0.89292. El mejor envío de todos sigue siendo E8.